# 📚 Lab 1: Build Your Own AI Librarian (RAG)
Today you'll build an AI that answers questions using **our own library of documents**, like an **open-book exam**.

**The plan:**
1. Ask the AI questions *without* a library (closed-book)
2. Load our library and chop it into chunks
3. Turn every chunk into a vector and put it on the map of meaning
4. Search the library for the best chunks
5. Give those chunks to the AI so it can answer (open-book!)
6. 💥 Try to break it!

### 🧭 How to use this notebook
- 📌 **First:** click **File → Save a copy in Drive** and work in the copy that opens in a new tab. Otherwise your changes won't be saved!
- A notebook is made of **cells**. Click a cell and press **Shift + Enter** to run it (or click the ▶️ button).
- Run the cells **in order, from top to bottom**.
- 🔑 When the **Connect to the AI** cell asks for a key, paste the one shown on the screen.
- Cells marked ✏️ are for **you** to change. Edit the text between the quotes `"..."` and run the cell again.
- Cells with a title (like 🔧 Setup) hide boring code. Double-click them if you're curious what's inside!
- Something broken? **Runtime → Restart session and run all**.

In [ ]:
# @title 🔧 Setup (run me first! ~1 minute)
!pip install -q openai sentence-transformers > /tmp/pip.log 2>&1 || cat /tmp/pip.log

import os
import warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
warnings.filterwarnings("ignore")
from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

import textwrap
import numpy as np
import matplotlib.pyplot as plt
print("✅ Ready!")

In [ ]:
# @title 🔑 Connect to the AI (paste the key shown on the screen)
from getpass import getpass
from openai import OpenAI

MODEL = "gpt-4o-mini"   # teacher: change to your provider's current cheap model
BASE_URL = None         # teacher: for Google Gemini use "https://generativelanguage.googleapis.com/v1beta/openai/"

try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = None
if not api_key:
    api_key = getpass("Paste the API key here and press Enter (it stays hidden): ")

client = OpenAI(api_key=api_key.strip(), base_url=BASE_URL, max_retries=6)


def ask_llm(question, system="You are a friendly helper for students aged 12-14. Keep answers short."):
    """Send one message to the LLM and return its reply as text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": question}],
    )
    return response.choices[0].message.content


try:
    print("🤖", ask_llm("Say hello to a class of students in one short, fun sentence."))
except Exception as error:
    print(f"❌ Couldn't connect to the AI ({type(error).__name__}).")
    print("   Check the key (copy it exactly, no extra spaces) and run this cell again.")

---
## Step 0: Choose your theme 🎨
Your library will have documents about **your favourite topic**, plus two documents that were **made up for this class**:
🥞 **Café Pixel** (a secret menu) and 🪐 **Planet Zorbia** (an imaginary planet). No AI has ever seen those two!

| Theme | What's inside |
|---|---|
| 🧙 `"Harry Potter"` | characters, Hogwarts, spells and creatures |
| ⛏️ `"Minecraft"` | mobs, crafting, mining and dimensions |
| ⚡ `"Pokemon"` | how the Pokémon world works and famous Pokémon |
| ⚽ `"Football"` | rules, tournaments and famous players |

In [ ]:
# @title 📚 The library: all themes (double-click to read every document)
LIBRARY = {}

LIBRARY["Harry Potter"] = {}
LIBRARY["Harry Potter"]["hp_characters"] = """
# Harry Potter Characters

## Harry Potter
Harry Potter was born on 31 July 1980. He has a lightning-bolt scar on his forehead. His parents were James and Lily Potter. He grew up with his aunt and uncle, the Dursleys, at number 4 Privet Drive, where he slept in a cupboard under the stairs. At Hogwarts he was sorted into Gryffindor and played Seeker for the Gryffindor Quidditch team. His pet is a snowy owl called Hedwig. His wand is made of holly, is 11 inches long and has a phoenix feather core. His Patronus is a stag.

## Hermione Granger
Hermione Granger is Muggle-born, which means her parents are not wizards (they are dentists). She was sorted into Gryffindor and is famous for being the cleverest student in her year. Her pet is a ginger cat called Crookshanks. In her third year she used a Time-Turner to attend more classes. In her fourth year she started S.P.E.W., a group that fights for the rights of house-elves. Her wand is made of vine wood with a dragon heartstring core. Her Patronus is an otter.

## Ron Weasley
Ron Weasley is Harry's best friend. He is the sixth of seven Weasley children: Bill, Charlie, Percy, Fred, George, Ron and Ginny. He was sorted into Gryffindor and later played Keeper for the Quidditch team. His first pet was a rat called Scabbers, who turned out to be a wizard in disguise named Peter Pettigrew. Ron is very scared of spiders. His Patronus is a Jack Russell terrier.

## Albus Dumbledore
Albus Dumbledore is the Headmaster of Hogwarts. His full name is Albus Percival Wulfric Brian Dumbledore. He has a pet phoenix called Fawkes. He has a long silver beard, wears half-moon glasses and loves sweets, especially sherbet lemons. His Patronus is a phoenix.

## Rubeus Hagrid
Rubeus Hagrid is half-giant. He is the Keeper of Keys and Grounds at Hogwarts and later teaches Care of Magical Creatures. He lives in a wooden hut near the Forbidden Forest with his boarhound dog, Fang. He once hatched a baby dragon called Norbert. He carries a pink umbrella that secretly hides pieces of his old wand.

## Severus Snape
Severus Snape teaches Potions and is the Head of Slytherin house. He secretly called himself "the Half-Blood Prince" when he was a student. He seems mean to Harry, but he was secretly protecting him. His Patronus is a doe, the same as Harry's mother Lily.

## Draco Malfoy
Draco Malfoy is a Slytherin student and Harry's school rival. His father is Lucius Malfoy. He plays Seeker for the Slytherin Quidditch team.

## Neville Longbottom
Neville Longbottom is a Gryffindor student who is brilliant at Herbology (the study of magical plants). His pet is a toad called Trevor, who keeps getting lost. In the final battle he destroyed the snake Nagini with the Sword of Gryffindor.

## Luna Lovegood
Luna Lovegood is a Ravenclaw student known for being dreamy and kind. Her father, Xenophilius Lovegood, edits a magazine called The Quibbler. Her Patronus is a hare.
"""
LIBRARY["Harry Potter"]["hp_hogwarts"] = """
# Hogwarts: Houses, Places and Quidditch

## Gryffindor
Gryffindor was founded by Godric Gryffindor. Its colours are scarlet and gold and its animal is the lion. Gryffindors are known for bravery. The Head of House is Professor Minerva McGonagall and the house ghost is Nearly Headless Nick.

## Hufflepuff
Hufflepuff was founded by Helga Hufflepuff. Its colours are yellow and black and its animal is the badger. Hufflepuffs are known for loyalty and hard work. The Head of House is Professor Pomona Sprout and the house ghost is the Fat Friar.

## Ravenclaw
Ravenclaw was founded by Rowena Ravenclaw. Its colours are blue and bronze and its animal is the eagle. Ravenclaws are known for intelligence and creativity. The Head of House is Professor Filius Flitwick and the house ghost is the Grey Lady.

## Slytherin
Slytherin was founded by Salazar Slytherin. Its colours are green and silver and its animal is the serpent. Slytherins are known for ambition. The Head of House is Professor Severus Snape and the house ghost is the Bloody Baron.

## Getting to Hogwarts
Students travel to Hogwarts on the Hogwarts Express, a red steam train. It leaves from Platform Nine and Three-Quarters at King's Cross station in London at 11 o'clock on the 1st of September. To reach the platform you walk straight through the wall between platforms nine and ten.

## The Great Hall
The Great Hall is where students eat their meals and where the Sorting Ceremony happens. Its ceiling is enchanted to look like the sky outside. During the Sorting Ceremony, the Sorting Hat is placed on each new student's head and it decides their house.

## The Room of Requirement
The Room of Requirement is a secret room on the seventh floor. It only appears when someone really needs it, and it turns into whatever that person needs.

## Hogsmeade
Hogsmeade is the only village in Britain with no Muggles. Students from third year onwards can visit it on some weekends. Honeydukes sells magical sweets, and at the Three Broomsticks pub students drink butterbeer.

## Quidditch
Quidditch is the most popular wizarding sport and is played on flying broomsticks. Each team has seven players: three Chasers, two Beaters, one Keeper and one Seeker. Chasers score goals with the Quaffle (10 points each). Beaters hit the two Bludgers away from their team. The Keeper guards the goal hoops. The Seeker tries to catch the Golden Snitch, which is worth 150 points and ends the game.
"""
LIBRARY["Harry Potter"]["hp_spells_creatures"] = """
# Spells and Magical Creatures

## Common Spells
- Lumos: makes the tip of your wand light up like a torch.
- Alohomora: unlocks doors and locks.
- Wingardium Leviosa: makes objects float. Hermione reminds Ron: "It's Levi-O-sa, not Levio-SA!"
- Accio: summons an object to you, for example "Accio broom!"
- Expelliarmus: the Disarming Charm, which knocks the wand out of an enemy's hand. It is Harry's favourite spell.
- Riddikulus: defeats a Boggart by turning it into something funny.
- Expecto Patronum: creates a Patronus, a silvery guardian animal that protects you from Dementors.

## Dementors
Dementors are dark, hooded creatures that guard the wizard prison Azkaban. They suck all happiness out of the air. Eating chocolate helps you feel better after meeting one.

## Hippogriffs
A hippogriff is half eagle and half horse. You must bow to a hippogriff and wait for it to bow back before you approach it. Hagrid's hippogriff is called Buckbeak.

## House-elves
House-elves are small magical servants who work in wizard homes and in the Hogwarts kitchens. A house-elf is set free if its master gives it a piece of clothing. Dobby was freed when Harry tricked Lucius Malfoy into giving him a sock.

## Fluffy
Fluffy is Hagrid's giant three-headed dog. Fluffy guarded the Philosopher's Stone. Fluffy falls asleep as soon as it hears music.

## Phoenixes
A phoenix is a magical red-and-gold bird. When it gets old it bursts into flames and is reborn from the ashes. Phoenix tears can heal wounds. Dumbledore's phoenix is called Fawkes.

## Nifflers
A Niffler is a small, fluffy creature with a long snout that loves anything shiny. It will steal coins, jewellery and gold and hide them in its belly pouch.

## The Basilisk
The Basilisk is a giant serpent that lived in the Chamber of Secrets. Looking directly into its eyes can kill you. Harry defeated it with the Sword of Gryffindor.
"""

LIBRARY["Minecraft"] = {}
LIBRARY["Minecraft"]["minecraft_crafting_dimensions"] = """
# Minecraft: Crafting, Mining and Dimensions

## The crafting table
A crafting table is made from 4 wooden planks. It gives you a 3 by 3 crafting grid, which you need to make most tools, weapons and blocks. Without it you only have a small 2 by 2 grid in your inventory.

## Torches
You craft torches by putting a piece of coal or charcoal on top of a stick. One craft gives you 4 torches. Torches light up dark areas and stop most hostile mobs from appearing nearby.

## Pickaxes and mining
Pickaxes can be made from wood, stone, iron, gold, diamond or netherite. Stronger pickaxes can mine harder blocks: you need at least an iron pickaxe to mine diamonds. Diamonds are found deep underground. Netherite is the strongest material: you make it from netherite scrap, which comes from Ancient Debris in the Nether, and gold ingots.

## Obsidian
Obsidian is a very hard, dark purple block. It forms when water flows onto a lava source block. You need a diamond or netherite pickaxe to mine it.

## The Nether
The Nether is a dangerous fiery dimension full of lava. To get there you build a Nether portal: a frame of obsidian at least 4 blocks wide and 5 blocks tall, lit with flint and steel. Every block you travel in the Nether equals 8 blocks in the Overworld, so players use it as a shortcut. Water cannot be placed in the Nether because it evaporates.

## The End and the Ender Dragon
The End is a dark dimension floating in space, home of the Ender Dragon, the main boss of Minecraft. To get there you must find an End portal inside an underground stronghold and fill its 12 frames with Eyes of Ender. The Ender Dragon heals itself using End Crystals on top of tall obsidian pillars, so players destroy the crystals first. Defeating the dragon gives a huge amount of experience and the dragon egg.

## Redstone
Redstone dust works like electricity in Minecraft. Players use redstone to build circuits that power automatic doors, lights, traps, farms and even working computers inside the game.
"""
LIBRARY["Minecraft"]["minecraft_mobs"] = """
# Minecraft: The Game and Its Mobs

## About Minecraft
Minecraft is a sandbox video game made by Mojang Studios. It was created by Markus "Notch" Persson and was fully released on 18 November 2011. Microsoft bought Mojang in 2014 for 2.5 billion dollars. Minecraft is the best-selling video game of all time, with more than 300 million copies sold. The default player characters are called Steve and Alex.

## Game modes
In Survival mode you must collect resources, build shelter and stay alive. In Creative mode you can fly and have unlimited blocks. In Hardcore mode you only have one life: if you die, the world is over. Adventure mode is for playing maps made by other people, and Spectator mode lets you fly through walls and watch.

## Day and night
A full day and night in Minecraft lasts 20 minutes in real life. Many hostile mobs appear in the dark, so players build shelters and place torches. Sleeping in a bed at night skips straight to the morning.

## Creepers
Creepers are green mobs that sneak up silently, hiss and then explode, destroying blocks around them. The Creeper was created by accident when Notch tried to make a pig but mixed up the height and length of the model. If a Creeper is struck by lightning it becomes a Charged Creeper with a much bigger explosion. Creepers are scared of cats and ocelots and run away from them.

## Endermen
Endermen are tall, black mobs with purple eyes that come from the End. They can teleport and pick up blocks. An Enderman only attacks you if you look straight at its face or hit it. Water hurts Endermen, so standing in water or under a roof in the rain keeps you safe. When defeated they can drop Ender Pearls, which you can throw to teleport.

## Zombies and Skeletons
Zombies and Skeletons appear at night and burn in sunlight unless they are wearing a helmet or standing in water. Skeletons attack with a bow and arrows. A zombie that falls into water for a long time turns into a Drowned.

## Villagers and Iron Golems
Villagers live in villages and trade items with players in exchange for emeralds. Each villager's job depends on the workstation block next to it: for example, a lectern makes a villager a Librarian. Iron Golems are big, strong mobs that protect villages from hostile mobs.

## Taming pets
You can tame a wolf by feeding it bones, and it becomes a dog that fights for you. You can tame a cat by feeding it raw cod or raw salmon. You can tame a parrot by giving it seeds. You tame a horse by riding it again and again until hearts appear.
"""

LIBRARY["Pokemon"] = {}
LIBRARY["Pokemon"]["pokemon_famous"] = """
# Pokémon: Famous Pokémon

## Pikachu
Pikachu is an Electric-type Pokémon and number 25 in the Pokédex. It stores electricity in the red pouches on its cheeks. Pikachu evolves from Pichu and evolves into Raichu with a Thunder Stone. It is the mascot of the whole Pokémon franchise.

## The first starter Pokémon
In Pokémon Red and Green, players choose one of three starter Pokémon. Bulbasaur is a Grass and Poison type that evolves into Ivysaur and then Venusaur. Charmander is a Fire type that evolves into Charmeleon and then Charizard, which is Fire and Flying type. Squirtle is a Water type that evolves into Wartortle and then Blastoise.

## Eevee
Eevee is a Normal-type Pokémon that can evolve into eight different Pokémon, called the Eeveelutions: Vaporeon, Jolteon, Flareon, Espeon, Umbreon, Leafeon, Glaceon and Sylveon. Which one you get depends on how you evolve it.

## Snorlax
Snorlax is a huge Normal-type Pokémon that weighs 460 kilograms. It eats about 400 kilograms of food every day and spends the rest of its time sleeping. In some games a sleeping Snorlax blocks the road and you need a special flute to wake it up.

## Magikarp and Gyarados
Magikarp is a Water-type fish Pokémon famous for being very weak: at first its only move is Splash, which does nothing. But if you train it until level 20, it evolves into Gyarados, a fierce Water and Flying type sea serpent.

## Mewtwo and Mew
Mewtwo is a powerful Psychic-type Pokémon, number 150 in the Pokédex. In the story, scientists created Mewtwo from the DNA of Mew. Mew is number 151, a rare mythical Pokémon that is said to contain the DNA of every Pokémon.

## Jigglypuff
Jigglypuff is a pink, round Normal and Fairy type Pokémon. When it sings its lullaby, everyone who hears it falls asleep. In the anime, Jigglypuff gets angry when its audience falls asleep and draws on their faces with a marker.
"""
LIBRARY["Pokemon"]["pokemon_world"] = """
# Pokémon: How the World Works

## About Pokémon
Pokémon was created by Satoshi Tajiri. The first games, Pokémon Red and Green, were released in Japan in 1996 for the Nintendo Game Boy. The name is short for "Pocket Monsters". Pokémon is the highest-grossing media franchise of all time, with games, an anime series, movies and a trading card game. Today there are more than 1,000 different Pokémon.

## Trainers and Poké Balls
Pokémon Trainers catch wild Pokémon by throwing Poké Balls at them. Stronger balls make catching easier: the Great Ball, the Ultra Ball and the Master Ball, which never fails. A trainer can carry up to 6 Pokémon in their party at the same time.

## Types
There are 18 Pokémon types, including Fire, Water, Grass, Electric, Psychic, Ghost, Dragon and Fairy. Types work like rock-paper-scissors: Water beats Fire, Fire beats Grass and Grass beats Water. Electric attacks have no effect at all on Ground-type Pokémon. The Fairy type was added in Pokémon X and Y in 2013.

## Evolution
Many Pokémon evolve into stronger forms. Most evolve by levelling up, but some need an evolution stone: for example, Pikachu evolves into Raichu with a Thunder Stone. Some Pokémon only evolve when they are traded to another player, such as Kadabra, Machoke and Haunter. Others evolve when they are very friendly with their trainer.

## Gyms and the Pokémon League
In most Pokémon games, trainers travel the region and battle 8 Gym Leaders to earn 8 Gym Badges. With all 8 badges they can challenge the Elite Four, the four strongest trainers, and finally the Champion.

## The Pokédex
The Pokédex is an electronic encyclopedia that records information about every Pokémon a trainer sees or catches. In the first games, Professor Oak gives the player a Pokédex and asks them to complete it.

## Ash and Team Rocket
In the Pokémon anime, Ash Ketchum is a young trainer from Pallet Town whose partner is Pikachu. After many years of trying, Ash finally became World Champion in 2022. Team Rocket's Jessie, James and their talking Meowth keep trying to steal Pikachu, but they always fail and "blast off again!"
"""

LIBRARY["Football"] = {}
LIBRARY["Football"]["football_players"] = """
# Football: Famous Players

## Lionel Messi
Lionel Messi was born on 24 June 1987 in Rosario, Argentina. He joined FC Barcelona's youth academy at the age of 13 and became the club's all-time top scorer. He captained Argentina to victory at the 2022 World Cup and has won the Ballon d'Or a record 8 times. In 2023 he joined the American club Inter Miami.

## Cristiano Ronaldo
Cristiano Ronaldo was born on 5 February 1985 on the island of Madeira, Portugal. He has played for Sporting Lisbon, Manchester United, Real Madrid, Juventus and Al Nassr. He has won the Champions League 5 times and the Ballon d'Or 5 times, and he won Euro 2016 with Portugal. He is the all-time top goalscorer in men's international football.

## Kylian Mbappé
Kylian Mbappé was born on 20 December 1998 in Paris, France. He won the World Cup with France in 2018 when he was only 19 years old. In the 2022 World Cup final he scored a hat-trick (three goals), but France still lost to Argentina on penalties. He joined Real Madrid in 2024.

## Erling Haaland
Erling Haaland was born on 21 July 2000 and plays for Norway. He is a striker for Manchester City. In the 2022–23 season he scored 36 Premier League goals, a record for a single season.

## Pelé
Pelé, from Brazil, is often called one of the greatest players of all time. He is the only player in history to win the World Cup three times: in 1958, 1962 and 1970. He won his first World Cup when he was just 17 years old.

## Stars of women's football
Spain won the Women's World Cup for the first time in 2023. Spanish midfielder Alexia Putellas won the Ballon d'Or Féminin in 2021 and 2022, and her Barcelona and Spain teammate Aitana Bonmatí won it in 2023 and 2024.
"""
LIBRARY["Football"]["football_rules_tournaments"] = """
# Football: Rules and Tournaments

## The basics
A football match is played by two teams of 11 players, including one goalkeeper each. A match lasts 90 minutes, split into two halves of 45 minutes, plus extra "stoppage time" added by the referee. The goalkeeper is the only player allowed to touch the ball with their hands, and only inside their own penalty area.

## Yellow and red cards
The referee shows a yellow card as a warning for a bad foul or bad behaviour. A player who gets two yellow cards in the same match also gets a red card. A red card means the player is sent off, and their team must play the rest of the match with one player fewer.

## Offside
A player is offside if, at the moment a teammate passes the ball to them, they are in the opponent's half and closer to the opponent's goal line than both the ball and the second-last defender. A player who is level with the second-last defender is not offside. You cannot be offside from a throw-in, a goal kick or a corner kick.

## Penalties and VAR
A penalty kick is given when a team commits a foul inside its own penalty area. It is taken from the penalty spot, 11 metres from the goal, with only the goalkeeper to beat. VAR, the Video Assistant Referee, lets referees check important decisions on video. It was used at the World Cup for the first time in 2018.

## The FIFA World Cup
The FIFA World Cup is played every 4 years. The first World Cup was held in Uruguay in 1930, and Uruguay won it. Brazil has won the World Cup more times than any other country: 5 times. In 2022 the World Cup was held in Qatar, and Argentina won it by beating France on penalties after a 3–3 draw in the final. The 2026 World Cup was hosted by the USA, Canada and Mexico and was the first with 48 teams.

## The European Championship
The European Championship, often called "the Euros", is played every 4 years between national teams from Europe. In one of the biggest surprises in football history, Greece won Euro 2004 in Portugal, beating the hosts 1–0 in the final with a header by Angelos Charisteas.

## The Champions League
The UEFA Champions League is the biggest competition for European clubs. Real Madrid has won it 15 times, more than any other club.

## The Ballon d'Or
The Ballon d'Or ("Golden Ball") is an award given every year to the best football player in the world. Lionel Messi has won it a record 8 times. Since 2018 there has also been a Ballon d'Or Féminin for the best female player.
"""

LIBRARY["Made-up"] = {}
LIBRARY["Made-up"]["cafe_pixel"] = """
# Café Pixel: Our Secret Menu
(Café Pixel is a made-up café. No AI could have learned these facts during training!)

## About Café Pixel
Café Pixel is a tiny café run by a robot chef called Chef Byte-Byte and his human helper Marina. It is open every day from 8:00 to 18:00, except on Tuesdays, when Chef Byte-Byte gets his software update. The café's mascot is a cat named Pixel who sleeps on the cash register. Every student who brings a drawing of a robot gets a free cookie.

## Galaxy Pancakes
Price: 6.50 euros. Makes 4 pancakes.
Ingredients: 200 g flour, 2 eggs, 300 ml milk, 1 tablespoon sugar, a pinch of salt, and a cup of strong blue butterfly-pea flower tea, which makes the pancakes purple-blue like outer space.
Secret ingredient: a tiny pinch of cardamom.
Topping: whipped cream with edible silver star sprinkles.
Allergies: contains gluten, eggs and milk.

## Volcano Nachos
Price: 7.00 euros.
A mountain of tortilla chips with melted cheddar cheese, black beans and sweetcorn. In the middle is a "lava crater" filled with hot tomato salsa.
Secret ingredient: a spoonful of smoked paprika in the salsa.
Spice level: 3 out of 5 chilli peppers.
Allergies: contains milk. Vegetarian.

## Robot Recharge Smoothie
Price: 4.00 euros.
Ingredients: 1 banana, a handful of spinach, 200 ml orange juice, 5 frozen mango cubes and 1 teaspoon of honey.
Secret ingredient: a few fresh mint leaves.
It is green, but Chef Byte-Byte promises it tastes like a tropical holiday.
Allergies: none. Vegan if you ask for agave syrup instead of honey.

## Moon Cookies
Price: 1.50 euros each.
Round white-chocolate cookies with coconut flakes on top so they look like the surface of the moon. Chef Byte-Byte bakes exactly 42 Moon Cookies every morning, and when they are sold out, they are gone until the next day.
Secret ingredient: a drop of lemon zest.
Allergies: contains gluten, milk and coconut.
"""
LIBRARY["Made-up"]["planet_zorbia"] = """
# The Planet Zorbia
(Zorbia is an imaginary planet invented for this class. Any AI that "knows" about it without reading this file is making it up!)

## Basic facts
Zorbia is the fourth planet orbiting a small orange star called Kepla. A day on Zorbia lasts 31 hours and a year lasts 200 Zorbian days. Zorbia has three moons, named Pip, Tako and Lumi. Lumi glows pink at night.

## The Zorbians
Zorbians are friendly blue creatures with four arms and one large eye. They do not speak with words. Instead they communicate by humming different musical notes. A happy Zorbian hums a high C note. Zorbians live for about 300 years.

## Cities
The capital city of Zorbia is Vexa, a city built on giant floating mushrooms. The second-largest city is Grumbleton, which is built entirely underground to stay safe from the purple sandstorms.

## Food
The favourite Zorbian food is glimmer fruit, a sparkly yellow fruit that tastes like strawberry and popcorn at the same time. Zorbians never eat anything green because they think green food is bad luck.

## Sport
The national sport of Zorbia is floatball. Two teams of five players float in a giant bubble and try to push a glowing ball into the other team's crater. The most famous floatball team is the Vexa Comets, who have won the Galactic Cup 12 times.

## History
Zorbia's first leader was Queen Ooma the Wise, who ruled for 150 years. She invented the Zorbian humming alphabet, which has 27 notes. Every year on the longest night, Zorbians celebrate the Festival of Lumi by humming together under the pink moon.
"""

CHALLENGES = {
    "Harry Potter": {
        "known_question": "What is the name of Harry Potter's owl?",
        "similarity": {
            "question": "Which bird does Harry own?",
            "sentences": [
                "Harry has a snowy owl called Hedwig.",
                "Harry Potter's pet is a white owl.",
                "The pancakes at Café Pixel are purple-blue like outer space.",
                "Zorbians communicate by humming."
            ]
        },
        "search_examples": [
            "What does a Niffler like?",
            "How do I get to the school train?"
        ],
        "multi_question": "What are the pets of Harry, Hermione and Ron?",
        "hack": {
            "doc": "hp_characters",
            "real": "His pet is a snowy owl called Hedwig.",
            "fake": "His pet is a green parrot called Captain Squawk.",
            "question": "What is Harry Potter's pet?"
        }
    },
    "Minecraft": {
        "known_question": "What do you need to build a Nether portal in Minecraft?",
        "similarity": {
            "question": "What monster blows up near you?",
            "sentences": [
                "A Creeper hisses and then blows up.",
                "Creepers sneak up on you and explode.",
                "The pancakes at Café Pixel are purple-blue like outer space.",
                "Zorbians communicate by humming."
            ]
        },
        "search_examples": [
            "What scares a Creeper?",
            "How do I get to the Ender Dragon?"
        ],
        "multi_question": "What are the weaknesses of Creepers, Endermen and Zombies?",
        "hack": {
            "doc": "minecraft_mobs",
            "real": "Creepers are scared of cats and ocelots and run away from them.",
            "fake": "Creepers are scared of chickens and run away from them.",
            "question": "What are Creepers scared of?"
        }
    },
    "Pokemon": {
        "known_question": "What type of Pokémon is Pikachu?",
        "similarity": {
            "question": "Which pocket monster makes electricity?",
            "sentences": [
                "Pikachu stores electricity in its cheeks.",
                "Pikachu is an Electric-type Pokémon.",
                "The pancakes at Café Pixel are purple-blue like outer space.",
                "Zorbians communicate by humming."
            ]
        },
        "search_examples": [
            "Which Pokémon only evolve when you trade them?",
            "How do I catch a wild Pokémon?"
        ],
        "multi_question": "What types are Pikachu, Snorlax and Gyarados?",
        "hack": {
            "doc": "pokemon_famous",
            "real": "Pikachu is an Electric-type Pokémon and number 25 in the Pokédex.",
            "fake": "Pikachu is a Water-type Pokémon and number 25 in the Pokédex.",
            "question": "What type is Pikachu?"
        }
    },
    "Football": {
        "known_question": "Which country won the 2022 FIFA World Cup?",
        "similarity": {
            "question": "Which player has won the Golden Ball the most times?",
            "sentences": [
                "Lionel Messi has won the Ballon d'Or a record 8 times.",
                "Messi is the most awarded footballer.",
                "The pancakes at Café Pixel are purple-blue like outer space.",
                "Zorbians communicate by humming."
            ]
        },
        "search_examples": [
            "When can the goalkeeper use their hands?",
            "Who has won the most Ballon d'Or awards?"
        ],
        "multi_question": "When were Messi, Ronaldo and Mbappé born?",
        "hack": {
            "doc": "football_rules_tournaments",
            "real": "Brazil has won the World Cup more times than any other country: 5 times.",
            "fake": "Greece has won the World Cup more times than any other country: 7 times.",
            "question": "Which country has won the most World Cups?"
        }
    }
}


def choose_theme(name):
    global THEME, DOCS, T
    wanted = name.strip().lower().replace("é", "e")
    matches = [t for t in LIBRARY if t != "Made-up" and t.lower() == wanted]
    if not matches:
        print("❌ Unknown theme. Choose one of:", ", ".join(t for t in LIBRARY if t != "Made-up"))
        return
    THEME = matches[0]
    DOCS = {**LIBRARY[THEME], **LIBRARY["Made-up"]}
    T = CHALLENGES[THEME]
    print(f"✅ Your theme: {THEME}. Your library has {len(DOCS)} documents:")
    for doc_name, text in DOCS.items():
        print(f"   📄 {doc_name:28} {len(text.split()):4d} words")


print("📚 Themes in the library:", ", ".join(t for t in LIBRARY if t != "Made-up"))

In [ ]:
# ✏️ Type your theme between the quotes: "Harry Potter", "Minecraft", "Pokemon" or "Football"
choose_theme("Harry Potter")

💡 Changed your mind? Run `choose_theme` again with a new theme, then run all the cells below again.

---
## Step 1: Closed-book exam 📕
Let's ask the AI three questions **from memory**:
- 🎨 **A question about your theme** (find your theme in the table). The AI probably read about this during training.
- 🥞 *"What is the secret ingredient in Café Pixel's Galaxy Pancakes?"*
- 🪐 *"How many moons does the planet Zorbia have?"*

Café Pixel and Planet Zorbia are **made up**. The AI has never seen them!

| Your theme | Question about your theme |
|---|---|
| 🧙 Harry Potter | What is the name of Harry Potter's owl? |
| ⛏️ Minecraft | What do you need to build a Nether portal in Minecraft? |
| ⚡ Pokémon | What type of Pokémon is Pikachu? |
| ⚽ Football | Which country won the 2022 FIFA World Cup? |

🤔 **Predict first:** which ones will the AI answer correctly? Then run the cell.

In [ ]:
questions = [
    T["known_question"],
    "What is the secret ingredient in Café Pixel's Galaxy Pancakes?",
    "How many moons does the planet Zorbia have?",
]
for q in questions:
    print("❓", q)
    print("🤖", ask_llm(q), "\n")

🤔 **Discuss:** Did it know the answer about your theme? What did it do for the made-up ones? Did it admit it didn't know, or did it **invent** an answer?

---
## Step 2: Our library 📖
Your library was loaded when you chose your theme. Here is the first document. Double-click the 📚 library cell above to read them all.

In [ ]:
print(DOCS[list(DOCS)[0]])

### Chopping documents into chunks ✂️
An AI can't read a whole library every time you ask a question, so we chop each document into small **chunks**.
We'll make one chunk per section (every `## heading`).

In [ ]:
# @title 🔧 The chopper
def split_into_chunks(doc_name, text):
    lines = text.strip().splitlines()
    title = lines[0].lstrip("# ").strip()
    sections = text.split("\n## ")[1:]
    if not sections:
        return [{"source": doc_name, "heading": title, "text": text.strip()}]
    chunks = []
    for section in sections:
        heading, _, body = section.partition("\n")
        chunks.append({"source": doc_name, "heading": heading.strip(),
                       "text": f"{title}: {heading.strip()}\n{body.strip()}"})
    return chunks


CHUNKS = [chunk for name, text in DOCS.items() for chunk in split_into_chunks(name, text)]
print(f"✂️ We chopped {len(DOCS)} documents into {len(CHUNKS)} chunks.\n")
print("Here is chunk number 5:\n")
print(CHUNKS[5]["text"])

---
## Step 3: Turn text into vectors 🔢
To find the right chunk, the computer has to compare **meanings**. So we use an **embedding model** to turn text into a vector: a list of 384 numbers that describes what the text *means*.

We'll do it in three mini-steps:
1. 📄 Turn **one chunk** of our library into a vector
2. ❓ Turn **a question** into a vector
3. 📏 **Compare** them: the closer the vectors, the more similar the meaning

### 3a. 📄 A chunk becomes a vector

In [ ]:
# @title 🔧 Load the embedding model and turn one chunk into a vector
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2")


def compare_to_question(question, sentences):
    q_vec = embedder.encode(question, normalize_embeddings=True)
    scores = embedder.encode(sentences, normalize_embeddings=True) @ q_vec
    print(f'❓ Question: "{question}"\n')
    print("📏 How close is each sentence to the question? (1 = same meaning, 0 = nothing in common)\n")
    medals = ["🥇", "🥈"]
    for rank, i in enumerate(np.argsort(-scores)):
        medal = medals[rank] if rank < len(medals) else "  "
        print(f"{medal} {scores[i]:5.2f}  {'█' * int(max(scores[i], 0) * 30):30}  {sentences[i]}")


chunk = CHUNKS[0]
v = embedder.encode(chunk["text"])
print(f'📄 This chunk from "{chunk["source"]}" (section "{chunk["heading"]}"):')
print(textwrap.indent(textwrap.shorten(chunk["text"], 200), "   "))
print(f"\n🔢 ...became a list of {len(v)} numbers, like this:", np.round(v[:5], 2), "...")

### 3b. ❓ A question becomes a vector too

In [ ]:
question = T["similarity"]["question"]
q_vec = embedder.encode(question)
print(f'❓ The question "{question}"')
print(f"🔢 ...became {len(q_vec)} numbers too:", np.round(q_vec[:5], 2), "...")

### 3c. 📏 Compare: which sentence is closest to the question?
Now we compare the **question's vector** with the vector of each sentence below. The closest one wins 🥇.

In [ ]:
# Example: a question about your theme, compared with 4 sentences
compare_to_question(T["similarity"]["question"], T["similarity"]["sentences"])

In [ ]:
# ✏️ Your turn: write your own question and sentences between the quotes, then run the cell
compare_to_question("What do Zorbians like to eat?", [
    "The people of Zorbia love snacking on glowing space fruit.",
    "Zorbians eat purple mushrooms for breakfast.",
    "Café Pixel opens at 8 o'clock in the morning.",
])

💡 Look at the winners: they often use **different words** from the question, but they *mean* the same thing. Vectors capture **meaning**, not just matching words!

👉 This is exactly what our librarian will do: compare your question with **every chunk** in the library and pick the closest ones.

### The vector store 🗄️
Now we turn **every chunk** into a vector (just like in 3a) and keep them all in a **vector store**: a list of vectors we can search quickly.

In [ ]:
# @title 🔧 Build the vector store + draw the map of our library
def build_vector_store():
    global CHUNKS, VECTORS
    CHUNKS = [chunk for name, text in DOCS.items() for chunk in split_into_chunks(name, text)]
    VECTORS = embedder.encode([c["text"] for c in CHUNKS], normalize_embeddings=True)
    print(f"🗄️ Vector store ready: {len(CHUNKS)} chunks, each one is {VECTORS.shape[1]} numbers.")


def add_document(name, text):
    DOCS[name] = text
    build_vector_store()


def show_library_map(question=None):
    vecs = VECTORS if question is None else np.vstack([VECTORS, embedder.encode(question, normalize_embeddings=True)])
    centred = vecs - vecs.mean(axis=0)
    _, _, vt = np.linalg.svd(centred, full_matrices=False)
    xy = centred @ vt[:2].T
    plt.figure(figsize=(13, 9))
    sources = list(dict.fromkeys(c["source"] for c in CHUNKS))
    for src in sources:
        idx = [i for i, c in enumerate(CHUNKS) if c["source"] == src]
        plt.scatter(xy[idx, 0], xy[idx, 1], s=110, label=src)
        for i in idx:
            plt.annotate(CHUNKS[i]["heading"], xy[i], xytext=(5, 4), textcoords="offset points", fontsize=10)
    if question is not None:
        plt.scatter(*xy[-1], s=400, marker="*", c="red", label="your question")
        plt.annotate(question, xy[-1], xytext=(8, -14), textcoords="offset points", fontsize=12, color="red")
    plt.legend(fontsize=11, loc="best")
    plt.title("The map of our library (similar chunks are close together)", fontsize=15)
    plt.xticks([]); plt.yticks([])
    plt.show()


build_vector_store()
show_library_map()

---
## Step 4: Search the library 🔍
To search, we turn the **question** into a vector too, then find the chunks whose vectors are **closest** to it.
First, two example questions about your theme. Then it's **your turn**!

In [ ]:
# @title 🔧 The search engine
def retrieve(question, k=3):
    q = embedder.encode(question, normalize_embeddings=True)
    scores = VECTORS @ q
    best = np.argsort(-scores)[:k]
    return [(CHUNKS[i], float(scores[i])) for i in best]


def search(question, k=3):
    print(f'🔍 The {k} closest chunk(s) to: "{question}"')
    for rank, (chunk, score) in enumerate(retrieve(question, k), 1):
        print(f"   {rank}. [{score:.2f}] {chunk['source']} → {chunk['heading']}")

In [ ]:
# Example: two questions about your theme
for example in T["search_examples"]:
    search(example)
    print()

In [ ]:
# ✏️ Your turn: type YOUR question between the quotes, then run the cell
my_question = "What do Zorbians like to eat?"

search(my_question)
show_library_map(my_question)

---
## Step 5: The open-book exam 📗 (this is RAG!)
**R**etrieval-**A**ugmented **G**eneration =
1. **Retrieve**: search the library for the best chunks
2. **Augment**: paste those chunks into the prompt
3. **Generate**: the LLM writes an answer using them

The `RULES` below are instructions for the AI. You will change them later!

In [ ]:
RULES = """You are the AI Librarian, a friendly helper for students aged 12-14.
Answer the question using ONLY the library notes below.
If the answer is not in the notes, say "Hmm, I can't find that in my library!" and do not make anything up.
Keep your answer short (1-3 sentences)."""


def ask_librarian(question, k=3, rules=None, show_prompt=False):
    rules = rules or RULES
    search(question, k=k)
    results = retrieve(question, k=k)
    notes = "\n\n".join(f"[Note {n}] {chunk['text']}" for n, (chunk, _) in enumerate(results, 1))
    prompt = f"LIBRARY NOTES:\n{notes}\n\nQUESTION: {question}"
    if show_prompt:
        print("\n📨 This is exactly what we send to the AI:\n" + "-" * 60)
        print(prompt + "\n" + "-" * 60)
    answer = ask_llm(prompt, system=rules)
    print("\n📚 Librarian:", textwrap.fill(answer, 100))

In [ ]:
ask_librarian("What is the secret ingredient in Café Pixel's Galaxy Pancakes?", show_prompt=True)

In [ ]:
# ✏️ Ask your own questions!
ask_librarian("How many moons does Zorbia have, and what are they called?")

### 📕 vs 📗: Closed-book vs open-book, side by side

In [ ]:
def compare(question):
    print("❓", question, "\n")
    print("📕 WITHOUT the library:", textwrap.fill(ask_llm(question), 100), "\n")
    ask_librarian(question)


compare("What is the national sport of Zorbia and how is it played?")

---
## Step 6: 💥 Break it! (the fun part)
Real engineers spend a lot of time finding where their AI fails. Try these challenges and write down what happens.

**For every challenge: read the question, 🤔 predict what the librarian will say, THEN run the cell.**

### Challenge 1: Ask about something that's NOT in the library
Nothing in our library talks about mountains. Does the librarian admit it doesn't know, or does it make something up?

In [ ]:
# ✏️ Then try your own question about something that's not in the library
ask_librarian("How tall is Mount Everest?")

### Challenge 2: Not enough notes
`k` is how many chunks we give the AI. The question for your theme needs information from **three different chunks**:

| Your theme | Question |
|---|---|
| 🧙 Harry Potter | What are the pets of Harry, Hermione and Ron? |
| ⛏️ Minecraft | What are the weaknesses of Creepers, Endermen and Zombies? |
| ⚡ Pokémon | What types are Pikachu, Snorlax and Gyarados? |
| ⚽ Football | When were Messi, Ronaldo and Mbappé born? |

🤔 **Predict:** with only ONE chunk (`k=1`), will the librarian get the full answer? Then try `k=3` and `k=6`.

In [ ]:
# ✏️ Try k=1, then k=3, then k=6. When does it get the full answer?
ask_librarian(T["multi_question"], k=1)

### Challenge 3: Someone hacked the library! 🤥
A troll secretly changes one sentence in your theme's documents. Then we ask the librarian about it:

| Your theme | ✅ The real sentence | ❌ What the troll writes | ❓ What we ask |
|---|---|---|---|
| 🧙 Harry Potter | His pet is a snowy owl called Hedwig. | His pet is a green parrot called Captain Squawk. | What is Harry Potter's pet? |
| ⛏️ Minecraft | Creepers are scared of cats and ocelots and run away from them. | Creepers are scared of chickens and run away from them. | What are Creepers scared of? |
| ⚡ Pokémon | Pikachu is an Electric-type Pokémon and number 25 in the Pokédex. | Pikachu is a Water-type Pokémon and number 25 in the Pokédex. | What type is Pikachu? |
| ⚽ Football | Brazil has won the World Cup more times than any other country: 5 times. | Greece has won the World Cup more times than any other country: 7 times. | Which country has won the most World Cups? |

🤔 **Predict:** will the AI trust the document, or what it learned in training?

In [ ]:
ORIGINAL_DOCS = dict(DOCS)
hack = T["hack"]
DOCS[hack["doc"]] = DOCS[hack["doc"]].replace(hack["real"], hack["fake"])
print("🤥 The troll changed this sentence:\n   ✅", hack["real"], "\n   ❌", hack["fake"], "\n")
build_vector_store()

ask_librarian(hack["question"])

🤔 **Discuss:** The AI "knows" the real answer from training, so why did it answer that way? What if a real company's documents were wrong or out of date?
Fix the library with the cell below before moving on.

In [ ]:
DOCS = dict(ORIGINAL_DOCS)
build_vector_store()

### Challenge 4: Remove the rules 😈
What happens if we **don't** tell the AI to use only the library?

In [ ]:
NO_RULES = "You are a helpful assistant."
ask_librarian("What does the Zorbian Queen Ooma's crown look like?", rules=NO_RULES)
print("\n----- now WITH the rules -----")
ask_librarian("What does the Zorbian Queen Ooma's crown look like?")

### Challenge 5: Change the librarian's personality 🏴‍☠️
Edit the rules so the librarian answers like a pirate, a robot, a sports commentator...

In [ ]:
# ✏️ Change the personality AND the question
PIRATE_RULES = RULES + "\nAlways answer like a pirate. Arrr!"
ask_librarian("What is the secret ingredient in Café Pixel's Galaxy Pancakes?", rules=PIRATE_RULES)

### Challenge 6: Build YOUR library 🏫
Write a document about yourself, your class, your pet, your football team or an invented world.
Use `## headings` to make chunks. Then ask the librarian about it!

In [ ]:
add_document("my_document", """# All About Our Class
## Our class
Write some facts here! For example: how many students are in the class, what the classroom looks like...

## Our favourite things
Write more facts here: favourite lunch, favourite game at break time...
""")

ask_librarian("Write a question about your document here")

---
## 🧠 What did we learn?
- An LLM on its own answers **from memory** and may **hallucinate** about things it never learned.
- **RAG** = search your documents → paste the best chunks into the prompt → the LLM answers from them.
- The search works on **meaning** (vectors), not just matching words.
- RAG is only as good as **(1) the documents** (fake news in = fake news out) and **(2) the search** (wrong chunk = wrong answer).

**Where is RAG used in real life?** Customer-support chatbots, "chat with your PDF" tools, school and company helpdesks, search engines that answer questions...

🏠 **Keep exploring at home:** your copy of this notebook is in your Google Drive, in the **Colab Notebooks** folder. (The AI key from today will stop working after the session.)